### 3. ✅ Validate Input with Type Hints — (Production-grade guardrails)
**Why it impresses**: uses inspect + typing modules, and the key move — tell the interviewer:

> "in FastAPI I'd use Pydantic for this — this shows the underlying concept."

In [ ]:
import functools
import inspect
from typing import get_type_hints

def validate_types(func):
    """
    Validates function arguments against type annotations at runtime.
    Real use case: service layer boundaries, API input sanitization.
    Note: Pydantic does this better in FastAPI — say that in interview!
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        hints = get_type_hints(func)
        sig   = inspect.signature(func)
        bound = sig.bind(*args, **kwargs)
        bound.apply_defaults()

        for param_name, value in bound.arguments.items():
            if param_name in hints:
                expected_type = hints[param_name]
                if not isinstance(value, expected_type):
                    raise TypeError(
                        f"Argument '{param_name}' expected "
                        f"{expected_type.__name__}, "
                        f"got {type(value).__name__}"
                    )
        return func(*args, **kwargs)
    return wrapper


# --- Usage ---
@validate_types
def create_order(user_id: int, amount: float, description: str):
    print(f"Order created: {description} — ${amount} for user {user_id}")

create_order(1, 99.99, "MacBook Pro")          # ✅
create_order("one", 99.99, "MacBook Pro")      # ❌ TypeError: user_id expected int